# 02 — Kat-İçi Ön İşleme, Manuel/Otomatik HP Araması ve 3×3 Nested CV

Bu notebook yalnızca **development** verisini kullanır. Test kümesi okunmaz.
Adaylar (M1–M6, R1–R6) bu notebook çalıştırılmadan önce `configs/` altında sabitlenip commit edilmiştir.
Outer değerlendirme skorları hiçbir seçimde kullanılmaz; nihai seçim tüm development üzerindeki iç CV ile yapılır.

## 0. Kurulum

In [ ]:
# Google Colab'da çalıştırılıyorsa aşağıdaki iki satırın başındaki '#' kaldırılır:
# !git clone https://github.com/mesutissever/ysa-midterm-mlp-nested-cv.git
# %cd ysa-midterm-mlp-nested-cv
# (Ham veri: data/bank-additional.csv — README'deki indirme talimatına bakınız)

In [ ]:
import sys, os, json, platform
sys.path.insert(0, os.path.abspath("."))
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import torch, sklearn

from src.data import load_seeds, get_dev_test, TARGET
from src.configs import load_candidates, load_json
from src.preprocessing import _leak_check
from src.model import expected_params
from src.nested_cv import run_nested_cv, METRIC_COLS

pd.set_option("display.max_columns", 60); pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid")
FIG, RES = "figures", "results"
def savefig(name): plt.savefig(f"{FIG}/{name}.png", dpi=150, bbox_inches="tight")

seeds = load_seeds()
df_dev, _ = get_dev_test()          # test kümesi kullanılmıyor
cands = load_candidates()
print("Development:", df_dev.shape, "| pozitif oranı:", round(df_dev[TARGET].mean(), 4))
print(f"Python {platform.python_version()} | torch {torch.__version__} | sklearn {sklearn.__version__} | "
      f"{platform.platform()} | {platform.processor()} | CUDA: {torch.cuda.is_available()}")
print("Seeds:", seeds)

## 1. Kat-içi ön işleme: sızıntı kontrolü

In [ ]:
_leak_check()

## 2. Sabitlenmiş adaylar ve eğitim ayarları

In [ ]:
training = load_json("training.json")
print(json.dumps(training, ensure_ascii=False, indent=1))
cand_tbl = pd.DataFrame([*cands["manual"].values(), *cands["random"].values()]).set_index("code")
cand_tbl["hidden"] = cand_tbl["hidden"].apply(lambda h: str(tuple(h)))
cand_tbl["params (d=61)"] = [expected_params(61, eval(h)) for h in cand_tbl["hidden"]]
cand_tbl.drop(columns="purpose").fillna("")

## 3. Nested CV'nin çalıştırılması

- Outer `StratifiedKFold(3, shuffle=True, random_state=31042)`, inner `StratifiedKFold(3, shuffle=True, random_state=31043)`.
- Her eğitimde: %85 fit (ön işleme fit + ağırlıklar) / %15 early-stop (BCE, scheduler) — seed 31044; ağırlık seed'i 31046.
- Bütçe: 3×3×12 = 108 iç eğitim + 6 refit + 36 development seçim eğitimi = **150 MLP eğitimi**.

`RUN = True` ise deney baştan çalıştırılır ve `results/` altına yazılır (CPU'da yaklaşık 5–10 dk).
`RUN = False` ise kayıtlı sonuçlar okunur.

In [ ]:
RUN = True
if RUN:
    out = run_nested_cv(df_dev, cands, seeds, save=True, verbose=True)

## 4. Sonuçların okunması

In [ ]:
inner  = pd.read_csv(f"{RES}/inner_cv_results.csv", keep_default_na=False)
manual = pd.read_csv(f"{RES}/manual_search_results.csv")
rand   = pd.read_csv(f"{RES}/random_search_results.csv")
outer  = pd.read_csv(f"{RES}/outer_cv_results.csv", keep_default_na=False)
summ   = pd.read_csv(f"{RES}/outer_cv_summary_formatted.csv", index_col=0)
sel    = pd.read_csv(f"{RES}/outer_selections.csv")
hist   = pd.read_csv(f"{RES}/training_histories.csv")
dev_sel = json.load(open(f"{RES}/dev_selection.json", encoding="utf-8"))
print("İç eğitim sayısı:", len(inner), "| toplam süre (dk):", round(dev_sel["total_runtime_s"] / 60, 1))

### Tablo 4b — Manuel adaylar (M1–M6): iç CV mean AP ± SD, süre, en iyi epoch

In [ ]:
cols = ["scope", "code", "mean_AP", "sd_AP", "mean_ROC_AUC", "mean_time_s", "mean_best_epoch",
        "mean_stopped_epoch", "n_params", "n_warnings", "optimizer", "hidden", "scaler", "lr", "scheduler"]
manual[cols].round(4)

### Tablo 5c — Otomatik adaylar (R1–R6): iç CV mean AP ± SD, süre, en iyi epoch

In [ ]:
cols_r = cols[:10] + ["optimizer", "hidden", "activation", "lr", "weight_decay", "batch_size", "scaler", "dropout", "clip"]
rand[cols_r].round(4)

### Her dış katta seçilen adaylar

In [ ]:
sel

### Tablo 6 — Dış kat bazlı sonuçlar (manuel, otomatik, referans modeller)

In [ ]:
outer[["outer_fold", "method", "selected"] + METRIC_COLS + ["TP", "FP", "TN", "FN", "train_time_s"]].round(4)

### Tablo 7 — Dış CV Mean ± SD (n = 3, örnek SD)

In [ ]:
summ[["AP", "ROC_AUC", "F1", "Sensitivity", "Specificity", "Balanced_Accuracy", "Precision", "Accuracy", "train_time_s"]]

*Not:* Üç dış kat birbirinden bağımsız tekrarlar değildir (eğitim kümeleri örtüşür). SD bir güven aralığı gibi yorumlanmamalıdır;
n = 3 nedeniyle belirsizlik sınırlı temsil edilir. Dummy modeli tüm örneklere negatif dediğinden Precision/Sensitivity/F1 = 0 ve
AP ≈ pozitif oranıdır (alt sınır).

### Tüm development üzerinde iç CV → nihai seçim

In [ ]:
dev_tbl = pd.concat([manual[manual.scope == "development"], rand[rand.scope == "development"]])
display(dev_tbl[["code", "mean_AP", "sd_AP", "mean_ROC_AUC", "mean_time_s", "mean_best_epoch", "n_params"]]
        .sort_values("mean_AP", ascending=False).round(4))
print(json.dumps({k: v for k, v in dev_sel.items() if k != "final_config"}, ensure_ascii=False, indent=1))
print("Nihai konfigürasyon:", dev_sel["final_config"])

### Uyarılar ve optimizer bazında süreler

In [ ]:
w = inner[inner["warnings"] != ""]
print(f"Uyarılı iç eğitim: {len(w)} / {len(inner)}  | başarısız: {int(inner['failed'].astype(str).eq('True').sum())}")
display(w.groupby(["code", "warnings"]).size().rename("n").reset_index())
cfg_all = {**cands["manual"], **cands["random"]}
inner["optimizer"] = inner["code"].map(lambda c: cfg_all[c]["optimizer"])
inner.groupby("optimizer")[["train_time_s", "best_epoch", "stopped_epoch"]].agg(["mean", "std"]).round(2)

## 5. Şekiller

### Şekil 1 — Veri bölünmesi ve nested CV akış şeması

In [ ]:
fig, ax = plt.subplots(figsize=(13, 7.5)); ax.axis("off")
def box(x, y, w, h, text, fc, fs=9, bold=False):
    ax.add_patch(plt.Rectangle((x, y), w, h, fc=fc, ec="#333", lw=1.2, zorder=1))
    ax.text(x + w / 2, y + h / 2, text, ha="center", va="center", fontsize=fs, zorder=3,
            weight="bold" if bold else "normal", linespacing=1.4)
def arrow(x1, y1, x2, y2):
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1), zorder=2,
                arrowprops=dict(arrowstyle="-|>", lw=1.4, color="#333", shrinkA=0, shrinkB=0))
box(0.36, 0.91, 0.28, 0.07, "bank-additional.csv (4.119 gözlem)\n'duration' çıkarıldı", "#e8e8e8")
box(0.06, 0.76, 0.40, 0.07, "Development %80 (3.295)\nEDA + tüm geliştirme kararları", "#cfe0f3")
box(0.62, 0.76, 0.32, 0.07, "Bağımsız test %20 (824)\nKİLİTLİ — yalnızca en sonda", "#f6d5c4")
arrow(0.45, 0.91, 0.26, 0.83); arrow(0.55, 0.91, 0.78, 0.83)
# Outer CV bloğu
box(0.01, 0.47, 0.58, 0.23, "", "#eef4fb")
ax.text(0.02, 0.675, "Outer StratifiedKFold — 3 kat (seed 31042)", fontsize=9.5, weight="bold", zorder=3)
box(0.03, 0.49, 0.33, 0.16, "OUTER-TRAIN\nInner StratifiedKFold — 3 kat (seed 31043)\n12 aday × 3 iç kat\nM1–M6 ve R1–R6 ayrı arama\nseçim: iç CV mean AP", "#ffffff", fs=8.5)
box(0.40, 0.49, 0.17, 0.16, "OUTER-EVAL\nmanuel kazanan\notomatik kazanan\nLogReg, Dummy\nyalnızca ÖLÇÜM", "#ffffff", fs=8.5)
arrow(0.36, 0.57, 0.40, 0.57)
arrow(0.26, 0.76, 0.26, 0.70)
# Bir eğitimin içi
box(0.03, 0.22, 0.54, 0.17, "Her MLP eğitiminde (inner-train veya outer-train):\n%85 FIT → ön işleme fit + ağırlık güncellemesi\n%15 EARLY-STOP → BCE kaybı: early stopping (15) + ReduceLROnPlateau (5)\nDeğerlendirme katı → yalnızca transform, AP", "#fffbe6", fs=8.5)
arrow(0.195, 0.49, 0.195, 0.39)
# Nihai seçim ve test
box(0.64, 0.49, 0.30, 0.16, "Tüm development üzerinde iç CV\n3 kat × 12 aday\n→ nihai konfigürasyon\n(en yüksek mean AP)", "#dcefdc", fs=8.5)
arrow(0.40, 0.76, 0.70, 0.65)
box(0.64, 0.22, 0.30, 0.17, "Nihai eğitim (3 seed)\n→ etiketsiz test tahminleri\n    ayrı commit\n→ ancak sonra test metrikleri", "#f6d5c4", fs=8.5)
arrow(0.79, 0.49, 0.79, 0.39)
ax.set_xlim(0, 1); ax.set_ylim(0.18, 1.0)
savefig("flow_nested_cv"); plt.show()

### Şekil 4 — Eğitim ve iç early-stop doğrulama kaybı; epoch bazlı öğrenme oranı (dış kat 1, iç kat 1)

In [ ]:
show = ["M1", "M3", "M6", dev_sel["random_best"]]
fig, axes = plt.subplots(2, len(show), figsize=(4.2 * len(show), 7), sharex=False)
for j, c in enumerate(show):
    h = hist[(hist.stage == "inner") & (hist.outer_fold == 1) & (hist.inner_fold == 1) & (hist.code == c)]
    best = h.loc[h.val_loss.idxmin(), "epoch"]
    axes[0, j].plot(h.epoch, h.train_loss, label="eğitim (fit)")
    axes[0, j].plot(h.epoch, h.val_loss, label="early-stop doğrulama")
    axes[0, j].axvline(best, ls="--", c="gray", lw=1); axes[0, j].set_title(f"{c} — en iyi epoch {best}")
    axes[0, j].set_ylabel("BCE kaybı"); axes[0, j].legend(fontsize=8)
    axes[1, j].step(h.epoch, h.lr, where="post", c="#d9822b"); axes[1, j].set_yscale("log")
    axes[1, j].set_xlabel("epoch"); axes[1, j].set_ylabel("öğrenme oranı")
plt.tight_layout(); savefig("training_curves"); plt.show()

### Şekil 5 — Manuel ve otomatik adayların iç CV AP karşılaştırması

In [ ]:
both = pd.concat([manual.assign(search="Manuel"), rand.assign(search="Otomatik")])
order = list(cands["manual"]) + list(cands["random"])
scopes = ["outer_1", "outer_2", "outer_3", "development"]
fig, ax = plt.subplots(figsize=(12, 4.8))
offs = np.linspace(-0.27, 0.27, len(scopes))
for k, sc in enumerate(scopes):
    d = both[both.scope == sc].set_index("code").reindex(order)
    ax.errorbar(np.arange(len(order)) + offs[k], d.mean_AP, yerr=d.sd_AP, fmt="o", capsize=3, ms=5,
                label=sc.replace("outer_", "dış kat ").replace("development", "tüm development"))
ax.axvline(5.5, c="gray", lw=1, ls=":")
ax.axhline(df_dev[TARGET].mean(), c="red", lw=1, ls="--", label="rastgele (pozitif oranı)")
ax.set_xticks(range(len(order))); ax.set_xticklabels(order)
ax.set_ylabel("iç CV mean AP ± SD"); ax.set_title("Manuel (M1–M6) ve otomatik (R1–R6) adaylar")
ax.legend(fontsize=8, ncol=3); plt.tight_layout(); savefig("search_comparison"); plt.show()

In [ ]:
# Dış CV: yöntemlerin AP karşılaştırması (kat bazlı noktalar)
fig, ax = plt.subplots(figsize=(7, 4))
sns.stripplot(data=outer, x="method", y="AP", hue="outer_fold", s=8, ax=ax, palette="deep")
sns.pointplot(data=outer, x="method", y="AP", errorbar="sd", color="black", markers="_", linestyles="none", ax=ax)
ax.set_title("Dış CV AP (nokta: kat; siyah: ortalama ± SD)"); ax.set_xlabel("")
plt.tight_layout(); savefig("outer_cv_ap"); plt.show()

## 6. Yorum (doldurulacak)

- H1–H4 hipotezlerinin iç CV sonuçlarına göre değerlendirilmesi (M1 vs M2/M3/M4/M5/M6; R adayları).
- Dış katlarda seçilen adayların tutarlılığı; manuel ve otomatik aramanın dış CV genelleme tahmini; LogReg/Dummy karşılaştırması.
- Nihai seçim: tüm development iç CV — dış sonuçlara bakılarak yapılmamıştır.